<a href="https://colab.research.google.com/github/omsoom/Multiplicative-SG/blob/main/Thalamus_Net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# MODEL 1: Instance Memory (Multiplicative 1/distance Gate)
# Proves: Direct-Jump Routing & Multiplicative Structural Gating works.
# ==========================================
class ThalamusMultiplicative(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_seq):
        B, T = x_seq.shape
        outputs = []
        memory_bank = []

        for t in range(T):
            curr_emb = self.embedding(x_seq[:, t])
            if t == 0:
                context = torch.zeros_like(curr_emb)
            else:
                past_embs = torch.stack(memory_bank, dim=1)
                distances = torch.arange(t, 0, -1, device=x_seq.device, dtype=torch.float32)
                # Multiplicative Structural Gate (1/distance)
                weights = 1.0 / distances
                weights = weights / weights.sum()
                w = weights.view(1, t, 1).expand(B, t, past_embs.size(-1))
                context = (w * past_embs).sum(dim=1)

            combined = torch.cat([curr_emb, context], dim=1)
            out = self.fc_out(self.relu(self.fc1(combined)))
            outputs.append(out)
            memory_bank.append(curr_emb)
        return torch.stack(outputs, dim=1)

# ==========================================
# MODEL 2: Instance Memory (Additive Semantic Attention)
# Proves: Additive/Learned Attention causes "semantic bleed" (throute).
# ==========================================
class ThalamusAdditiveAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.query_layer = nn.Linear(embed_dim, embed_dim)
        self.key_layer = nn.Linear(embed_dim, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_seq):
        B, T = x_seq.shape
        outputs = []
        memory_bank = []

        for t in range(T):
            curr_emb = self.embedding(x_seq[:, t])
            if t == 0:
                context = torch.zeros_like(curr_emb)
            else:
                past_embs = torch.stack(memory_bank, dim=1)
                # Standard Additive Semantic Attention
                query = self.query_layer(curr_emb).unsqueeze(1)
                keys = self.key_layer(past_embs)
                semantic_scores = torch.matmul(query, keys.transpose(1, 2)).squeeze(1) / (self.embedding.embedding_dim ** 0.5)
                weights = F.softmax(semantic_scores, dim=1)
                w = weights.unsqueeze(-1)
                context = (w * past_embs).sum(dim=1)

            combined = torch.cat([curr_emb, context], dim=1)
            out = self.fc_out(self.relu(self.fc1(combined)))
            outputs.append(out)
            memory_bank.append(curr_emb)
        return torch.stack(outputs, dim=1)

# ==========================================
# MODEL 3: Fixed V x V Graph (No Surprise Gate)
# Proves: "Type vs. Instance" overwriting problem (Banana Problem).
# ==========================================
class ThalamusFixedGraphNoGate(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, gamma=0.95):
        super().__init__()
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_seq):
        B, T = x_seq.shape
        device = x_seq.device
        dynamic_graph = torch.zeros(B, self.vocab_size, self.embedding.embedding_dim, device=device)
        outputs = []

        for t in range(T):
            x_t = x_seq[:, t]
            curr_emb = self.embedding(x_t)
            context_row = dynamic_graph[torch.arange(B, device=device), x_t, :]

            combined = torch.cat([curr_emb, context_row], dim=1)
            out = self.fc_out(self.relu(self.fc1(combined)))
            outputs.append(out)

            # Static decay and update (Overwrites types)
            dynamic_graph = dynamic_graph * self.gamma
            dynamic_graph[torch.arange(B, device=device), x_t, :] += curr_emb

        return torch.stack(outputs, dim=1)

# ==========================================
# MODEL 4: Fixed V x V Graph (Surprise Gate)
# Proves: Runaway feedback loop failure.
# ==========================================
class ThalamusFixedGraphSurprise(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, gamma=0.95, alpha=1.0):
        super().__init__()
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.alpha = alpha
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_seq):
        B, T = x_seq.shape
        device = x_seq.device
        dynamic_graph = torch.zeros(B, self.vocab_size, self.embedding.embedding_dim, device=device)
        outputs = []
        prev_logits = None

        for t in range(T):
            x_t = x_seq[:, t]
            curr_emb = self.embedding(x_t)
            context_row = dynamic_graph[torch.arange(B, device=device), x_t, :]

            combined = torch.cat([curr_emb, context_row], dim=1)
            curr_logits = self.fc_out(self.relu(self.fc1(combined)))
            outputs.append(curr_logits)

            if prev_logits is not None:
                prob = F.softmax(prev_logits, dim=-1)[0, x_t.item()].item()
                surprise = -math.log(prob + 1e-8)
                gate = 1.0 + (self.alpha * surprise)
            else:
                gate = 1.0

            dynamic_graph = dynamic_graph * self.gamma
            update_vector = curr_emb * gate
            dynamic_graph[torch.arange(B, device=device), x_t, :] += update_vector

            prev_logits = curr_logits

        return torch.stack(outputs, dim=1)

# ==========================================
# RUN ABLATION STUDY
# ==========================================
def train_and_eval(model, name, input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000):
    print(f"\n{'='*60}")
    print(f"TESTING: {name}")
    print(f"{'='*60}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(input_seq)
        loss = criterion(outputs.view(-1, vocab_size), target_seq.view(-1))
        loss.backward()
        optimizer.step()
        if (epoch+1) % 200 == 0:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        seed = "the brain"
        gen_ids = [vocab[c] for c in seed]
        for _ in range(len(text) - len(seed)):
            out = model(torch.tensor([gen_ids]))
            next_id = torch.argmax(out[0, -1, :]).item()
            gen_ids.append(next_id)
        gen_text = "".join([idx_to_char[i] for i in gen_ids])

    print(f"\nFinal Loss: {loss.item():.4f}")
    print(f"Generated:  {gen_text}")
    print(f"Target:     {text}")
    match = "PASS (100% Accuracy)" if gen_text == text else "FAIL"
    print(f"Result:     {match}")
    return loss.item(), gen_text

# Setup Data
text = "the brain processes information using neurons and synapses. the thalamus routes the signal through the network. it is a beautiful system."
chars = sorted(list(set(text)))
vocab = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

input_seq = torch.tensor([[vocab[c] for c in text[:-1]]])
target_seq = torch.tensor([[vocab[c] for c in text[1:]]])

print(f"Vocab: {vocab_size}, Seq Len: {len(text)}")

# Run all 4 models to prove the 4 points
m1 = ThalamusMultiplicative(vocab_size, 64, 128)
train_and_eval(m1, "Model 1: Multiplicative Structural Gate (1/distance)", input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000)

m2 = ThalamusAdditiveAttention(vocab_size, 64, 128)
train_and_eval(m2, "Model 2: Additive Semantic Attention (Learned)", input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000)

m3 = ThalamusFixedGraphNoGate(vocab_size, 64, 128, gamma=0.95)
train_and_eval(m3, "Model 3: Fixed VxV Graph (No Surprise Gate)", input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000)

m4 = ThalamusFixedGraphSurprise(vocab_size, 64, 128, gamma=0.95, alpha=1.0)
train_and_eval(m4, "Model 4: Fixed VxV Graph (Surprise Gate)", input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000)

Vocab: 23, Seq Len: 137

TESTING: Model 1: Multiplicative Structural Gate (1/distance)
Epoch 200, Loss: 0.0009
Epoch 400, Loss: 0.0003
Epoch 600, Loss: 0.0001
Epoch 800, Loss: 0.0001
Epoch 1000, Loss: 0.0000

Final Loss: 0.0000
Generated:  the brain processes information using neurons and synapses. the thalamus routes the signal through the network. it is a beautiful system.
Target:     the brain processes information using neurons and synapses. the thalamus routes the signal through the network. it is a beautiful system.
Result:     PASS (100% Accuracy)

TESTING: Model 2: Additive Semantic Attention (Learned)
Epoch 200, Loss: 0.3974
Epoch 400, Loss: 0.3361
Epoch 600, Loss: 0.1451
Epoch 800, Loss: 0.1169
Epoch 1000, Loss: 0.1585

Final Loss: 0.1585
Generated:  the brain processes information using urons and synapses. the thalamus througnal through nethetwork. is is is is is is isys. is. is. is. 
Target:     the brain processes information using neurons and synapses. the thalamus routes

(0.7407879829406738,
 'the brain processes inforong us thesynghes tighes thrork. th thes s s bes baigh ba balal bamamusys s s bauroces bal bamutign s s bal baut')

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# MODEL 1: Instance Memory + Direct-Jump + Multiplicative Gating
# Proves Points 1, 2, and 3.
# ==========================================
class ThalamusInstanceGraph(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_seq):
        B, T = x_seq.shape
        outputs = []
        memory_bank = [] # Point 1: Explicit topological memory

        for t in range(T):
            curr_emb = self.embedding(x_seq[:, t])
            if t == 0:
                context = torch.zeros_like(curr_emb)
            else:
                past_embs = torch.stack(memory_bank, dim=1)

                # Point 2: Direct-Jump Retrieval (accessing all past instances)
                # Point 3: Multiplicative Structural Gating (1/distance)
                distances = torch.arange(t, 0, -1, device=x_seq.device, dtype=torch.float32)
                weights = 1.0 / distances
                weights = weights / weights.sum()

                w = weights.view(1, t, 1).expand(B, t, past_embs.size(-1))
                context = (w * past_embs).sum(dim=1)

            combined = torch.cat([curr_emb, context], dim=1)
            out = self.fc_out(self.relu(self.fc1(combined)))
            outputs.append(out)
            memory_bank.append(curr_emb)
        return torch.stack(outputs, dim=1)

# ==========================================
# MODEL 2: Fixed V x V Token-Type Graph
# Proves Point 4: Token Types != Token Instances (Fails)
# ==========================================
class ThalamusFixedTypeGraph(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, gamma=0.95):
        super().__init__()
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x_seq):
        B, T = x_seq.shape
        device = x_seq.device
        # Fixed V x V matrix (Token Types)
        dynamic_graph = torch.zeros(B, self.vocab_size, self.embedding.embedding_dim, device=device)
        outputs = []

        for t in range(T):
            x_t = x_seq[:, t]
            curr_emb = self.embedding(x_t)
            context_row = dynamic_graph[torch.arange(B, device=device), x_t, :]

            combined = torch.cat([curr_emb, context_row], dim=1)
            out = self.fc_out(self.relu(self.fc1(combined)))
            outputs.append(out)

            # Update fixed matrix (Overwrites instances)
            dynamic_graph = dynamic_graph * self.gamma
            dynamic_graph[torch.arange(B, device=device), x_t, :] += curr_emb

        return torch.stack(outputs, dim=1)

# ==========================================
# RUN ABLATION STUDY
# ==========================================
def train_and_eval(model, name, input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000):
    print(f"\n{'='*60}")
    print(f"TESTING: {name}")
    print(f"{'='*60}")

    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(input_seq)
        loss = criterion(outputs.view(-1, vocab_size), target_seq.view(-1))
        loss.backward()
        optimizer.step()
        if (epoch+1) % 200 == 0:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        seed = "the brain"
        gen_ids = [vocab[c] for c in seed]
        for _ in range(len(text) - len(seed)):
            out = model(torch.tensor([gen_ids]))
            next_id = torch.argmax(out[0, -1, :]).item()
            gen_ids.append(next_id)
        gen_text = "".join([idx_to_char[i] for i in gen_ids])

    print(f"\nFinal Loss: {loss.item():.4f}")
    print(f"Generated:  {gen_text}")
    print(f"Target:     {text}")
    match = "PASS (100% Accuracy)" if gen_text == text else "FAIL"
    print(f"Result:     {match}")
    return loss.item(), gen_text

# Setup Data
text = "the brain processes information using neurons and synapses. the thalamus routes the signal through the network. it is a beautiful system."
chars = sorted(list(set(text)))
vocab = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

input_seq = torch.tensor([[vocab[c] for c in text[:-1]]])
target_seq = torch.tensor([[vocab[c] for c in text[1:]]])

print(f"Vocab: {vocab_size}, Seq Len: {len(text)}")

# Prove Points 1, 2, 3: Instance Memory + Direct-Jump + Multiplicative Gating
m1 = ThalamusInstanceGraph(vocab_size, 64, 128)
train_and_eval(m1, "Model 1: Instance Graph (Direct-Jump + Mult. Gating)", input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000)

# Prove Point 4: Fixed Token-Type Graph Fails
m2 = ThalamusFixedTypeGraph(vocab_size, 64, 128, gamma=0.95)
train_and_eval(m2, "Model 2: Fixed VxV Token-Type Graph", input_seq, target_seq, vocab_size, idx_to_char, text, epochs=1000)

Vocab: 23, Seq Len: 137

TESTING: Model 1: Instance Graph (Direct-Jump + Mult. Gating)
Epoch 200, Loss: 0.0009
Epoch 400, Loss: 0.0003
Epoch 600, Loss: 0.0001
Epoch 800, Loss: 0.0001
Epoch 1000, Loss: 0.0000

Final Loss: 0.0000
Generated:  the brain processes information using neurons and synapses. the thalamus routes the signal through the network. it is a beautiful system.
Target:     the brain processes information using neurons and synapses. the thalamus routes the signal through the network. it is a beautiful system.
Result:     PASS (100% Accuracy)

TESTING: Model 2: Fixed VxV Token-Type Graph
Epoch 200, Loss: 0.3469
Epoch 400, Loss: 0.2389
Epoch 600, Loss: 0.2033
Epoch 800, Loss: 0.1860
Epoch 1000, Loss: 0.2606

Final Loss: 0.2606
Generated:  the brain processes information us alamuron amul tiong thes thesion tinghrone th t at beapronghrmus th sit syn bes besious. s. bes.heth i
Target:     the brain processes information using neurons and synapses. the thalamus routes the signal

(0.26061081886291504,
 'the brain processes information us alamuron amul tiong thes thesion tinghrone th t at beapronghrmus th sit syn bes besious. s. bes.heth i')